NOTICE: Code is optimized for the A100 GPU

# 1. Installing, importing libraries and making configurations

In [ ]:
!pip install uv

In [ ]:
!uv pip install torch numpy matplotlib lightning huggingface_hub triton pandas optuna optuna-integration[pytorch_lightning]

In [ ]:
!uv pip install mamba-ssm --no-build-isolation-package mamba-ssm

In [ ]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


import lightning as L
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, login
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

from mamba_ssm import Mamba

import optuna
from optuna.integration import PyTorchLightningPruningCallback

In [ ]:
NUM_WORKERS = 4

NUM_EPOCHS = 5

# 2. Models developed

In [ ]:
def label_smoothed_bce_loss(logits, labels, eps=0.1):
    ce_hard = F.cross_entropy(logits, labels)
    log_probs = F.log_softmax(logits, dim=-1)
    ce_soft = -log_probs.mean(dim=-1).mean()
    return (1.0 - eps) * ce_hard + eps * ce_soft

In [ ]:
def get_optimizer(model, lr=1e-3, base_wd=0.01, dt_proj_wd=1e-3):
    dt_proj_weights = []
    other_params    = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'dt_proj.weight' in name:
            dt_proj_weights.append(param)
        else:
            other_params.append(param)
    param_groups = [
        {'params': other_params,    'weight_decay': base_wd,    'name': 'default'},
        {'params': dt_proj_weights, 'weight_decay': dt_proj_wd, 'name': 'dt_proj'},
    ]
    return torch.optim.AdamW(param_groups, lr=lr)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, expand=2, **kwargs):
        super().__init__()
        self.norm  = RMSNorm(d_model)
        self.mamba = Mamba(d_model=d_model, d_state=N, expand=expand)

    def forward(self, x):
        return x + self.mamba(self.norm(x))

In [ ]:
class MambaClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 2,
        N: int = 16,
        num_classes: int = 2,
        lr: float = 1e-3,
        label_smoothing: float = 0.0,
        dt_proj_wd: float = 0.0,
        expand: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.label_smoothing = label_smoothing
        self.dt_proj_wd = dt_proj_wd
        self.num_classes = num_classes

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        # Compile Mamba layers for fused Triton kernels (replaces causal-conv1d)
        for i, layer in enumerate(self.layers):
            self.layers[i] = torch.compile(layer)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)               # (B, L, D)

        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)

        return self.classifier(x)

    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch['attention_mask'])
        labels = batch['labels']

        if self.label_smoothing > 0:
            loss = label_smoothed_bce_loss(logits, labels, eps=self.label_smoothing)
        else:
            loss = F.cross_entropy(logits, labels)

        preds = logits.argmax(dim=-1)
        acc = (preds == labels).float().mean()

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc',  acc,  prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, 'test')

    def configure_optimizers(self):
        if self.dt_proj_wd > 0:
            return get_optimizer(
                self, lr=self.lr, base_wd=0.01, dt_proj_wd=self.dt_proj_wd
            )
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)

# 3. Data and running each model in its benchmark

## 3.0 Data

In [ ]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt']

for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

In [ ]:
def make_trainer():
    return L.Trainer(
        max_epochs=6,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        logger=CSVLogger(save_dir="logs", name="lra-benchmarks"),
    )

In [ ]:
def make_hpo_trainer(max_epochs=9):
    """Lightweight trainer for HPO — no checkpointing, no logging."""
    return L.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
    )

## 3.1 ListOpsDataset

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_listops = torch.load('listops.pt', weights_only=False)

### 3.1.1 Finding the bets hyperparameters

In [ ]:
listops_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}

def objective(trial):
    d_model    = trial.suggest_categorical("d_model",    listops_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   listops_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          listops_search_space["N"])
    lr         = trial.suggest_categorical("lr",         listops_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", listops_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_listops['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    try:
        trainer.fit(model, train_loader, val_loader)
        return trainer.callback_metrics["val_acc"].item()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise optuna.TrialPruned()

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(listops_search_space),
    study_name="listops-mamba-grid",
)

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study.optimize(objective, n_trials=2)

### 3.1.2 Training the model with the best hyperpaparemters

In [ ]:
best = study.best_trial.params

print(best)

In [ ]:
model_to_find_num_parameters = MambaClassifier(
    vocab_size=data_listops['vocab_size'],
    d_model=best["d_model"],
    n_layers=best["n_layers"],
    N=best["N"],
    num_classes=num_classes,
    lr=best["lr"],
)

total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study.best_trial.params)
print("Best val_acc:", study.best_trial.value)

print(f"Best val_acc: {study.best_trial.value:.4f} | Params: {study.best_trial.params}")

print(f"Model Parameters: {total_params}")

In [ ]:
# Datasets & loaders
train_dataset = ListOpsDataset(data_listops['train'].data)
val_dataset = ListOpsDataset(data_listops['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_listops['train'].data]
num_classes = max(train_labels) + 1

# Model
model_listops = MambaClassifier(
    vocab_size=data_listops['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_listops, train_loader, val_loader)

test_dataset = ListOpsDataset(data_listops['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_listops, test_loader)

## 3.2 IMDbDataset

In [ ]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_imdb = torch.load('imdb_lra.pt', weights_only=False)

### 3.2.1 Finding the bets hyperparameters

In [ ]:
imdb_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}


def objective_imdb(trial):
    d_model    = trial.suggest_categorical("d_model",    imdb_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   imdb_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          imdb_search_space["N"])
    lr         = trial.suggest_categorical("lr",         imdb_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", imdb_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_imdb['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_imdb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(imdb_search_space),
    study_name="imdb-mamba-grid",
)

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_imdb.optimize(objective_imdb, n_trials=2)

In [ ]:
best = study_imdb.best_trial.params

print(best)

model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_imdb['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_imdb.best_trial.params)
print("Best val_acc:", study_imdb.best_trial.value)

print(f"Best val_acc: {study_imdb.best_trial.value:.4f} | Params: {study_imdb.best_trial.params}")

print(f"Model Parameters: {total_params}")

### 3.2.2 Running the best model with the best hyperparameters

In [ ]:
# Datasets & loaders
train_dataset = IMDbDataset(data_imdb['train'].data)
val_dataset = IMDbDataset(data_imdb['val'].data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_imdb['train'].data]
num_classes = max(train_labels) + 1

# Model
model_imdb = MambaClassifier(
    vocab_size=data_imdb['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

In [ ]:
trainer = make_trainer()

In [ ]:
trainer.fit(model_imdb, train_loader, val_loader)

In [ ]:
test_dataset = IMDbDataset(data_imdb['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
trainer.test(model_imdb, test_loader)

## 3.3 ACLRetricalDataset

In [ ]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_retrieval = torch.load('acl_retrieval_lra.pt', weights_only=False)

### 3.3.1 Finding the bets hyperparameters

In [ ]:
retrieval_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [4, 8],
}

def objective_retrival(trial):
    d_model    = trial.suggest_categorical("d_model",    retrieval_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   retrieval_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          retrieval_search_space["N"])
    lr         = trial.suggest_categorical("lr",         retrieval_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", retrieval_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_retrieval['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_retrival = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(retrieval_search_space),
    study_name="retrieval-mamba-grid",
)

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_retrival.optimize(objective_retrival, n_trials=2)

In [ ]:
best = study_retrival.best_trial.params
#chnage this
model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_retrieval['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )

# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_retrival.best_trial.params)
print("Best val_acc:", study_retrival.best_trial.value)

print(f"Best val_acc: {study_retrival.best_trial.value:.4f} | Params: {study_retrival.best_trial.params}")

print(f"Model Parameters: {total_params}")

### 3.3.2 Training model with best hyperparameters

In [ ]:
# Datasets & loaders
train_dataset = ACLRetrievalDataset(data_retrieval['train'].data)
val_dataset = ACLRetrievalDataset(data_retrieval['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_retrieval['train'].data]
num_classes = max(train_labels) + 1

# Model
model_acl = MambaClassifier(
    vocab_size=data_retrieval['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_acl, train_loader, val_loader)

test_dataset = ACLRetrievalDataset(data_retrieval['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_acl, test_loader)

## 3.4 CIFAR10SequentialDataset

In [ ]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Handle both dict and tuple formats
        if isinstance(item, dict):
            tokens = item['input_ids']
            label = item['labels']
        else:
            tokens = item[0]
            label = item[-1]

        # Convert tensors to lists if needed (for padding logic)
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        if isinstance(label, torch.Tensor):
            label = label.item()

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
data_cifar10_sequential = torch.load('cifar10_sequential_lra.pt', weights_only=False)

### 3.4.1 Finding the bets hyperparameters

In [ ]:
cifar_search_space = {
    "d_model":    [64, 128],
    "n_layers":   [2, 4],
    "N":          [8, 16],
    "lr":         [1e-3, 5e-4],
    "batch_size": [8, 16],
}

def objective_cifar(trial):
    d_model    = trial.suggest_categorical("d_model",    cifar_search_space["d_model"])
    n_layers   = trial.suggest_categorical("n_layers",   cifar_search_space["n_layers"])
    N          = trial.suggest_categorical("N",          cifar_search_space["N"])
    lr         = trial.suggest_categorical("lr",         cifar_search_space["lr"])
    batch_size = trial.suggest_categorical("batch_size", cifar_search_space["batch_size"])

    model = MambaClassifier(
        vocab_size=data_cifar10_sequential['vocab_size'],
        d_model=d_model,
        n_layers=n_layers,
        N=N,
        num_classes=num_classes,
        lr=lr,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

    trainer = make_hpo_trainer(max_epochs=NUM_EPOCHS)
    trainer.fit(model, train_loader, val_loader)
    return trainer.callback_metrics["val_acc"].item()

In [ ]:
study_cifar = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(cifar_search_space),
    study_name="cifar-mamba-grid",
)

In [ ]:
# Datasets & loaders for Optuna study
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'].data)
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'].data)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_cifar10_sequential['train'].data]
num_classes = max(train_labels) + 1

In [ ]:
study_cifar.optimize(objective_cifar, n_trials=2)

In [ ]:
best = study_cifar.best_trial.params

#chnage this
model_to_find_num_parameters = MambaClassifier(
        vocab_size=data_cifar10_sequential['vocab_size'],
        d_model=best["d_model"],
        n_layers=best["n_layers"],
        N=best["N"],
        num_classes=num_classes,
        lr=best["lr"],
    )


# 2. Calculate total trainable parameters
total_params = sum(p.numel() for p in model_to_find_num_parameters.parameters() if p.requires_grad)

print("Best trial:", study_cifar.best_trial.params)
print("Best val_acc:", study_cifar.best_trial.value)

print(f"Best val_acc: {study_cifar.best_trial.value:.4f} | Params: {study_cifar.best_trial.params}")

print(f"Model Parameters: {total_params}")

### 3.4.2 Training model with best hyperparameters

In [ ]:
# Datasets & loaders
train_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['train'].data)
val_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['val'].data)

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

# Calculate num_classes from the training data
train_labels = [item[1] for item in data_cifar10_sequential['train'].data]
num_classes = max(train_labels) + 1

# Model
model_cifar = MambaClassifier(
    vocab_size=data_cifar10_sequential['vocab_size'],
    d_model=best['d_model'],
    n_layers=best['n_layers'],
    N=best['N'],
    num_classes=num_classes,
    lr=best['lr'],
)

trainer = make_trainer()

trainer.fit(model_cifar, train_loader, val_loader)

test_dataset = CIFAR10SequentialDataset(data_cifar10_sequential['test'].data)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

trainer.test(model_cifar, test_loader)

# 4. Seeing results

In [ ]:
print("\n" + "=" * 70)
print("  GRID SEARCH RESULTS SUMMARY (Optuna GridSampler)")
print("=" * 70)

for name, study_obj in [
    ("ListOps",     study),
    ("IMDb",        study_imdb),
    ("Retrieval",   study_retrival),
    ("CIFAR-10",    study_cifar),
]:
    print(f"\n  {name:15s}  best val_acc = {study_obj.best_trial.value:.4f}")
    print(f"  {'':15s}  params: {study_obj.best_trial.params}")

print("\n" + "=" * 70)